# `model_spatial.pt` on ten held-back frames

What the checkpoint actually predicts, drawn two ways:

* **2D** — the predicted body projected back into the crop it was cut from, over ASPset's own keypoints.
  This is the honest test: the network never saw these clips, and the overlay either lands on the person or it does not.
* **3D** — the same prediction as a posed MHR mesh, sitting at its predicted distance in front of the camera,
  with the true keypoints beside it. Drag to orbit.

Both figures have a slider over the ten examples.

Pipeline as a reminder: `prepare.py` wrote the crops and labels, `dataset.py` serves them,
`model_spatial.py` turns one crop into 49 numbers, `losses.py` poses the real rig from those numbers.
Nothing new is defined here that those files already provide — the 49 numbers become a body
through exactly the same code the training loop scored.

In [1]:
import pathlib, sys

import numpy as np
import plotly.graph_objects as go
import torch

HERE = pathlib.Path.cwd()
sys.path[:0] = [str(HERE), str(HERE.parent / "mhr_posing")]

import losses, mhr_scene, model_spatial
from dataset import IMAGENET_MEAN, IMAGENET_STD, PoseData, split_clips
from mhr_kit.model import faces
from prepare import KEYPOINTS, PELVIS, ik

CHECKPOINT = HERE / "model_spatial.pt"
EXAMPLES = 10
CROP_PIXELS = 256          # the size dataset.py serves, so crop units +/-1 span this

checkpoint = torch.load(CHECKPOINT, map_location="cpu")

# augment=False: no random zoom, shift or flip, so what is drawn is what the
# model was scored on rather than one particular roll of the dice.
_train_clips, val_clips = split_clips()
val = PoseData(val_clips, augment=False)

net = model_spatial.PoseNet(checkpoint["backbone"], pretrained=False)
net.load_state_dict(checkpoint["weights"])
net.eval()

# The rig, wrapped so it can be posed from the 49 numbers. CPU only, and that is
# fine: ten frames is one forward pass.
body = losses.Body(ik.Rig())
TRIANGLES = faces(body.rig.model)

print(f"{checkpoint['model']} / {checkpoint['backbone']}   "
      f"{sum(p.numel() for p in net.parameters()) / 1e6:.1f} M parameters")
print(f"{len(val_clips)} held-back clips, {len(val)} frames, {TRIANGLES.shape[0]} triangles")

spatial / resnet18   11.3 M parameters
10 held-back clips, 1310 frames, 1186 triangles


## The ten frames, and what the model says about them

Spread evenly across the held-back split rather than taken from its start, so the
sample covers several clips and both camera sides instead of ten near-identical
neighbours from one video.

Everything downstream is built here, once:

    view          (10, 17, 3)    the predicted body's keypoints, viewer frame, cm
    mesh          (10, 595, 3)   the same body's skin, same frame
    truth         (10, 17, 3)    where ASPset says the keypoints are

`view` is `losses.compute_loss`'s own three lines — pose the rig, work out where it
sits from `(u, v, size)`, put the two together — so what is drawn is the thing that
was trained, not a re-derivation of it.

In [2]:
chosen = np.linspace(0, len(val) - 1, EXAMPLES).astype(int)
batch = torch.utils.data.default_collate([val[int(i)] for i in chosen])

with torch.no_grad():
    prediction = net(batch["image"], batch["context"])

# The three lines from losses.compute_loss, in the same order.
local = body.joints(prediction["angles"], prediction["scales"])              # pose, hips at origin
translation = losses.translation_from(prediction["camera"], batch["box"], batch["intrinsics"])
view = losses.place_in_view(local, prediction["rotation"], translation).detach()

# The skin, posed identically. body.joints throws the vertices away because the
# loss never needs them; here they are the whole point. Both come out of one call,
# and subtracting the pelvis puts the vertices in the same hips-at-origin frame
# body.joints returns, so the same rotation and translation apply unchanged.
parameters = torch.zeros(EXAMPLES, losses.N_MHR_PARAMETERS)
parameters = parameters.index_copy(1, body.angles_at, prediction["angles"])
parameters = parameters.index_copy(1, body.scales_at, prediction["scales"])
with torch.no_grad():
    vertices, state = body.rig.model(torch.zeros(EXAMPLES, losses.N_IDENTITY), parameters, None)
hips = state[:, body.joints_at[PELVIS], :3]
mesh = losses.place_in_view(vertices - hips[:, None, :], prediction["rotation"],
                            translation).detach().numpy()

truth = batch["truth"]                     # ASPset's markers — what the scores are against

# Per-example MPJPE, hips aligned, exactly as losses.metrics defines it.
per_example_mm = (view - view[:, PELVIS:PELVIS + 1]
                  - (truth - truth[:, PELVIS:PELVIS + 1])).norm(dim=-1).mean(dim=1) * 10.0

scores = losses.metrics(prediction, batch, body)
print("   ".join(f"{name} {value:.1f}" for name, value in scores.items()))
print()
print(f"{'#':>2s}  {'clip':22s}{'frame':>7s}{'mpjpe':>8s}{'away':>8s}")
for n, index in enumerate(chosen):
    clip, row = val.index[index]
    print(f"{n:2d}  {clip:22s}{val.shards[clip]['frame'][row]:7d}"
          f"{float(per_example_mm[n]):7.0f}mm{-float(view[n, PELVIS, 2]) / 100:7.1f}m")

mpjpe_mm 73.0   pixel_error 10.4   root_error_cm 18.5   rotation_deg 7.1

 #  clip                    frame   mpjpe    away
 0  1e28-0001-left              0     52mm   17.6m
 1  1e28-0001-left            290     39mm   17.3m
 2  1e28-0021-mid             288     62mm   18.8m
 3  1e28-0038-left             60     92mm   19.0m
 4  1e28-0065-left             62     88mm   17.7m
 5  1e28-0080-right            48     84mm   18.8m
 6  8a59-0006-mid             158    116mm   18.2m
 7  8a59-0040-left             24     54mm   16.4m
 8  8a59-0071-right            98     91mm   15.9m
 9  8a59-0093-mid             150     53mm   20.7m


## 2D — the prediction back in the picture

The predicted body is projected through the same camera the photograph came from and
drawn over the crop. `losses.project` and `losses.to_crop_units` are the pair the 2D
loss already uses; crop units run -1 to +1 across the crop, so `(u + 1) / 2 * 256`
puts them on the 256-pixel image.

This view includes *position* error, unlike MPJPE — nothing is slid onto anything.
If the orange skeleton is the right shape but offset, `(u, v, size)` is what is wrong,
not the pose.

In [3]:
BONES = [("pelvis", "spine"), ("spine", "neck"), ("neck", "head"), ("head", "head_top"),
         ("neck", "left_shoulder"), ("left_shoulder", "left_elbow"), ("left_elbow", "left_wrist"),
         ("neck", "right_shoulder"), ("right_shoulder", "right_elbow"), ("right_elbow", "right_wrist"),
         ("pelvis", "left_hip"), ("left_hip", "left_knee"), ("left_knee", "left_ankle"),
         ("pelvis", "right_hip"), ("right_hip", "right_knee"), ("right_knee", "right_ankle")]
LINKS = [(KEYPOINTS.index(a), KEYPOINTS.index(b)) for a, b in BONES]


def stick(points):
    """(17, D) keypoints -> D lists, one per axis, ready for a single line trace.

    A None between bones breaks the line, so all sixteen bones are one trace
    instead of sixteen. That matters for the slider: a frame has to replace a
    fixed set of traces, and one trace per figure is far less to swap.
    """
    points = np.asarray(points)
    return [[value for a, b in LINKS for value in (points[a, axis], points[b, axis], None)]
            for axis in range(points.shape[1])]


def to_crop_pixels(points):
    """Viewer-frame points (B, N, 3) -> pixels in the 256-crop (B, N, 2)."""
    units = losses.to_crop_units(losses.project(points, batch["intrinsics"]), batch["box"])
    return ((units + 1.0) / 2.0 * CROP_PIXELS).numpy()


predicted_2d, truth_2d = to_crop_pixels(view), to_crop_pixels(truth)

# Undo the ImageNet normalisation dataset.py applied, and put the colour channel last.
pictures = np.clip(batch["image"].numpy().transpose(0, 2, 3, 1) * IMAGENET_STD + IMAGENET_MEAN,
                   0.0, 1.0) * 255.0


def overlay(n):
    """The three traces for example n: the crop, the truth, the prediction."""
    true_x, true_y = stick(truth_2d[n])
    predicted_x, predicted_y = stick(predicted_2d[n])
    return [
        mhr_scene._picture(pictures[n]),
        go.Scatter(x=true_x, y=true_y, mode="lines+markers", name="ASPset keypoints",
                   line=dict(color="deepskyblue", width=3), marker=dict(size=5)),
        go.Scatter(x=predicted_x, y=predicted_y, mode="lines+markers", name="predicted",
                   line=dict(color="orange", width=3), marker=dict(size=5)),
    ]


def caption(n):
    clip, _row = val.index[chosen[n]]
    return f"{n}  {clip}   {float(per_example_mm[n]):.0f} mm"


def slider(count, read):
    """One slider stepping a figure through `count` prebuilt frames."""
    return [dict(x=0.05, len=0.9, currentvalue=dict(prefix="example "),
                 steps=[dict(method="animate", label=str(n),
                             args=[[str(n)], dict(frame=dict(duration=0, redraw=True),
                                                  mode="immediate")])
                        for n in range(count)])]


figure = go.Figure(overlay(0))
figure.frames = [go.Frame(name=str(n), data=overlay(n),
                          layout=go.Layout(title=dict(text=caption(n))))
                 for n in range(EXAMPLES)]
figure.update_layout(
    height=560, width=620, margin=dict(l=0, r=0, t=50, b=40),
    title=dict(text=caption(0), x=0.02, font=dict(size=12)),
    xaxis=dict(visible=False, range=[0, CROP_PIXELS]),
    # autorange="reversed": image row 0 is the TOP, and a plotly y axis counts up.
    yaxis=dict(visible=False, range=[CROP_PIXELS, 0], scaleanchor="x"),
    legend=dict(orientation="h", y=1.02, x=0.0),
    sliders=slider(EXAMPLES, overlay))
figure

## 3D — the same body, where the model thinks it is

Now the mesh, in the viewer frame: camera at the origin looking down -z, so a body
sits at negative z exactly as far away as the model believes it is. `mhr_scene`
supplies the layout, including the one thing that is easy to get silently wrong —
plotting as `(x, -z, y)` rather than swapping two axes, which would mirror the whole
scene and leave a body that looks entirely plausible while facing the wrong way.

The blue skeleton is ASPset's; the grey mesh is the prediction. They will not coincide
exactly even for a perfect solve — MHR's joint centres sit inside the skin and ASPset's
markers sit on it, which is the ~16 mm floor `losses.py` prints.

Drag to orbit, scroll to zoom. The eye starts on the camera's side, so the first look
is roughly what the photograph above shows.

In [4]:
# One box big enough for every example, centred per example — so stepping the slider
# does not make the view leap about, and a distant frame is not a speck.
centres, reach = mhr_scene._framing([[mesh[n], truth[n].numpy()] for n in range(EXAMPLES)])


def scene(n):
    """The three traces for example n: the predicted skin, the truth, the predicted bones."""
    true_x, true_y, true_z = stick(mhr_scene.plotted(truth[n].numpy()))
    predicted_x, predicted_y, predicted_z = stick(mhr_scene.plotted(view[n].numpy()))
    return [
        mhr_scene._body(mesh[n], TRIANGLES, "lightsteelblue", "predicted body"),
        go.Scatter3d(x=true_x, y=true_y, z=true_z, mode="lines+markers",
                     name="ASPset keypoints", line=dict(color="deepskyblue", width=5),
                     marker=dict(size=3), hoverinfo="skip"),
        go.Scatter3d(x=predicted_x, y=predicted_y, z=predicted_z, mode="lines+markers",
                     name="predicted keypoints", line=dict(color="orange", width=5),
                     marker=dict(size=3), hoverinfo="skip"),
    ]


figure = go.Figure(scene(0))
figure.frames = [go.Frame(name=str(n), data=scene(n),
                          layout=go.Layout(title=dict(text=caption(n)),
                                           scene=mhr_scene._scene(centres[n], reach)))
                 for n in range(EXAMPLES)]
figure.update_layout(
    height=640, margin=dict(l=0, r=0, t=50, b=40),
    title=dict(text=caption(0), x=0.02, font=dict(size=12)),
    scene=mhr_scene._scene(centres[0], reach),
    legend=dict(orientation="h", y=0.02, x=0.0),
    sliders=slider(EXAMPLES, scene))

print(f"handedness {mhr_scene.handedness():+.0f}   (+1 means the scene is not mirrored)")
figure

handedness +1   (+1 means the scene is not mirrored)


## Where the error actually sits

One number per keypoint, hips aligned, averaged over the ten. The shape of this
column is the thing to look at rather than any single value.

In [5]:
column = (view - view[:, PELVIS:PELVIS + 1]
          - (truth - truth[:, PELVIS:PELVIS + 1])).norm(dim=-1).mean(dim=0) * 10.0

for name, millimetres in zip(KEYPOINTS, column):
    bar = "#" * int(float(millimetres) / 4)
    print(f"{name:16s}{float(millimetres):6.0f} mm  {bar}")

pelvis               0 mm  
spine               39 mm  #########
neck                46 mm  ###########
head                58 mm  ##############
head_top            93 mm  #######################
left_shoulder       65 mm  ################
left_elbow          87 mm  #####################
left_wrist         131 mm  ################################
right_shoulder      55 mm  #############
right_elbow         99 mm  ########################
right_wrist        160 mm  ########################################
left_hip            18 mm  ####
left_knee           71 mm  #################
left_ankle         126 mm  ###############################
right_hip           20 mm  #####
right_knee          66 mm  ################
right_ankle        107 mm  ##########################


## How to read the two together

| what you see | what it means |
| --- | --- |
| 2D overlay lands, 3D mesh sits at the wrong depth | `size` is off. A crop cannot see distance, and the 2D view is nearly blind to it — a body twice as far with twice the build projects identically. |
| 2D overlay offset but the right shape | `(u, v)` — where the hips sit in the crop — rather than the pose. |
| limbs right, torso leaning | the root rotation. Compare `rotation_deg` against the per-example MPJPE. |
| everything slightly short or long | the nine bone scales. Two subjects in ASPset, so build is the least varied thing in the labels and the easiest to underfit. |

Worth remembering what the split does and does not prove: whole clips are held back,
so these are unseen actions and camera angles — but both subjects appear on both sides
of the split, so none of this says the model handles a new person.